# **Neuromorphic computing - LAB 04**
May-June 2026, "Machine learning in applications" course

*Prof. G. Urgese, V. Fra, B. Leto*

---
This notebook is an assignment to be completed and submitted by Thursday 21 at 23:59. The objective is to perform a series of tasks on the miRNA dataset using Spiking Neural Networks (SNNs). Throughout the assignment, you will explore the dataset, apply the required preprocessing and analysis steps, and implement SNN-based methods to address the proposed tasks.
---

In [2]:
!pip install snntorch==0.6.2 --quiet

In [3]:
import torch, torch.nn as nn
import snntorch as snn

### Useful functions

In [4]:
import csv
import numpy as np
import scipy.stats
from sklearn.model_selection import train_test_split

In [5]:
def extract_label(file_name, verbose=False):
    data = {}
    label = []
    with open(file_name, "r") as fin:
        reader = csv.reader(fin, delimiter=',')
        first = True
        for row in reader:
            lbl = row[2]
            if first or "TARGET" in lbl:
                first = False
                continue
            lbl = lbl.replace("TCGA-","")

            label.append(lbl)
            if lbl in data.keys():
                data[lbl] += 1
            else:
                data[lbl] = 1
    if verbose:
        print(f"Number of classes in the dataset = {len(data)}")
        pprint.pprint(data, indent=4)

    return label

In [6]:
def create_dictionary(labels):
    dictionary = {}
    class_names = np.unique(labels)
    for i, name in enumerate(class_names):
        dictionary[name] = i
    return dictionary

In [7]:
def label_processing(labels):
    new_miRna_label = []
    dictionary = create_dictionary(labels)
    for i in labels:
        new_miRna_label.append(dictionary[i])
    return new_miRna_label

### 1.2 Download Dataset

In [8]:
import os
mir_dataset = "https://drive.google.com/drive/folders/1oWWeord8YYvtxIo2Pq2peyx7xOI-1Tmb?usp=sharing"
if "MLinApp_course_data" not in os.listdir("./"):
  ! gdown $mir_dataset -O ./MLinApp_course_data --folder

In [9]:
# Remove the first row and the last column from the feature
miR_label = extract_label("./MLinApp_course_data/tcga_mir_label.csv")
miR_data = np.genfromtxt('./MLinApp_course_data/tcga_mir_rpm.csv', delimiter=',')[1:,0:-1]

In [10]:
number_to_delete = abs(len(miR_label) - miR_data.shape[0])
miR_data = miR_data[number_to_delete:,:]
# Convert labels in number
num_miR_label = label_processing(miR_label)

In [11]:
# Z-score normalization
miR_data = scipy.stats.zscore(miR_data, axis=1)

assert np.isnan(miR_data).sum() == 0

In [11]:
print(miR_data[0], np.min(miR_data))

[ 1.68703834  1.67910068  1.71667838 ... -0.05112508 -0.01854857
  3.38106288] -0.13941802539632334


In [ ]:
# log2 normalization <Optional>

# miR_data = miR_data + abs(np.min(miR_data)) + 0.001

# miR_data = np.log2(miR_data)

In [ ]:
# normalization between [0, 255] <Optional>
# miR_data = (miR_data - np.min(miR_data)) / (np.max(miR_data) - np.min(miR_data)) * 255

In [12]:
n_classes = np.unique(miR_label).size

print(n_classes)
print(miR_label)

print(num_miR_label)
print(miR_data)

33
['READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'READ', 'REA

# DataLoading
Define variables for dataloading.

In [12]:
batch_size = 128
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
padded_data = False

Define function to add padding to our data \<Optional\>

In [13]:
import math

def add_pad_data(data):
  miR_data = data
  c_int = math.ceil(np.sqrt(len(miR_data[0])))
  pad = c_int ** 2 - len(miR_data[0])
  pad_width = (0, pad)

  padded_miR_data = np.zeros((miR_data.shape[0], miR_data.shape[1] + pad_width[1]))

  for i in range(len(miR_data)):
    padded_miR_data[i] = np.pad(miR_data[i], pad_width, mode='constant')

  # reshape shape[1] into (c_int, c_int)

  dim = int(np.sqrt(len(padded_miR_data[0])))
  padded_miR_data = padded_miR_data.reshape((padded_miR_data.shape[0],1, dim, dim))

  return padded_miR_data

## TODO: Generate subset based on top N most frequent labels \<Optional\>

From the dataset extract the 10 most frequent classes

In [14]:
N = 10


In [15]:
# TODO: Write here your the code for identifying the most frequent classes
unique, counts = np.unique(num_miR_label, return_counts=True)
most_frequent_classes = unique[np.argsort(counts)[::-1]][0:N]
print(most_frequent_classes)

[ 2 11 30 28  9 16 22 14 17 19]


In [16]:
# Fixed:
subset_data = miR_data[np.isin(num_miR_label, most_frequent_classes), :]
subset_label_raw = list(np.array(num_miR_label)[np.isin(num_miR_label, most_frequent_classes)])
subset_label = label_processing(subset_label_raw)   # re-maps to 0..9
n_classes = N

## TODO: Dimensionality analysis and reduction using Principal Component Analysis  \<Optional\>

---

(PCA) on train_data.

Keep only features that preserve 99% of the variance.

For further information, please look at the documentation available at https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html

In [17]:
from sklearn.decomposition import PCA

In [18]:
preserve_perc = 0.99

In [19]:
pca_reduced = PCA(n_components=preserve_perc)
# pca_train_data_reduced = pca_reduced.fit_transform(miR_data)
pca_train_data_reduced = pca_reduced.fit_transform(subset_data)

print("{}% variance with {} components out of {}".format(int(preserve_perc*100),pca_reduced.n_components_,subset_data.shape[1]))

99% variance with 26 components out of 1881


## Create DataLoader

In [ ]:
# usefull if you want to represent your data as images <Optional>

# subset_data = add_pad_data(subset_data)
# padded_data = True

In [20]:
# train_data, val_data, train_label, val_label = train_test_split(miR_data, num_miR_label, test_size=0.20, random_state=42)
train_data, val_data, train_label, val_label = train_test_split(pca_train_data_reduced, subset_label, test_size=0.20, random_state=42)
# train_data, val_data, train_label, val_label = train_test_split(subset_data, subset_label, test_size=0.20, random_state=42)

In [21]:
from torch.utils.data import TensorDataset, DataLoader
miR_train = torch.Tensor(train_data)
miR_train_label = torch.Tensor(train_label)
miR_dataset_train = TensorDataset(miR_train, miR_train_label)

miR_val = torch.Tensor(val_data)
miR_val_label = torch.Tensor(val_label)
miR_dataset_val = TensorDataset(miR_val, miR_val_label)

train_loader = DataLoader(miR_dataset_train, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(miR_dataset_val, batch_size=batch_size)

# Define Network
Let's compare the performance of a pair of networks both with and without population coding.


Each group should try the assigned values.


In [22]:
#TODO: Select a random configuration of parameters
from snntorch import surrogate

# network parameters
if padded_data:
  num_inputs = train_data.shape[2] ** 2
else:
  num_inputs = train_data.shape[1]

num_hidden = 256 #GROUPS : A: [64], B: [128], C: [256], D: [512]
num_outputs = n_classes

# temporal dynamics
num_steps = 20 #GROUPS : A: [5], B: [10], C: [20], D: [50]

# spiking neuron parameters
beta = 0.85  # neuron decay rate  #GROUPS : A: [0.7], B: [0.8], C: [0.85], D: [0.9 - 1]
grad = surrogate.fast_sigmoid()

## Without population coding

In [17]:
first_layer_neuron = snn.Leaky(beta=beta, spike_grad=grad, init_hidden=True)
second_layer_neuron =  snn.Leaky(beta=beta, spike_grad=grad, init_hidden=True, output=True)

In [ ]:
#TODO: change neuron type es. Lapique https://snntorch.readthedocs.io/en/latest/snn.neurons_lapicque.html <Optional>
# first_layer_neuron = snn.Lapicque(beta=beta, spike_grad=grad, init_hidden=True)
# second_layer_neuron =  snn.Lapicque(beta=beta, spike_grad=grad, init_hidden=True, output=True)

In [ ]:
#TODO: # change neuron type es. RLeaky https://snntorch.readthedocs.io/en/latest/snn.neurons_rleaky.html <Optional>
# first_layer_neuron = snn.RLeaky(beta=beta, spike_grad=grad, init_hidden=True)
# second_layer_neuron =  snn.RLeaky(beta=beta, spike_grad=grad, init_hidden=True, output=True)

In [18]:
# standard network
net = nn.Sequential(nn.Flatten(),
                    nn.Linear(num_inputs, num_hidden),
                    first_layer_neuron,
                    nn.Linear(num_hidden, num_outputs),
                    second_layer_neuron
                    ).to(device)

## Next Step: define your own network

In [ ]:
# TODO: define your own network (If you need another layer you need to define it. You can change the type of neurons between different layers)
# A full list of neurons type is available at https://snntorch.readthedocs.io/en/latest/snntorch.html#neuron-list
# Execute this step after reporting the results of the previous standard network

# net = ...

## With population coding


In [32]:
#TODO: Select a random configuration of parameters
neurons_per_classes = 75  #GROUPS : A: [25], B: [50], C: [75], D: [100]
pop_outputs = n_classes * neurons_per_classes

In [ ]:
# first_layer_neuron =  snn.Leaky(beta=beta, spike_grad=grad, init_hidden=True)
# second_layer_neuron =  snn.Leaky(beta=beta, spike_grad=grad, init_hidden=True, output=True)

In [38]:
# TODO: change neuron type es. Lapique https://snntorch.readthedocs.io/en/latest/snn.neurons_lapicque.html <Optional>

first_layer_neuron = snn.RLeaky(beta=beta, spike_grad=grad, all_to_all=False, init_hidden=True)
second_layer_neuron =  snn.RLeaky(beta=beta, spike_grad=grad, all_to_all=False, init_hidden=True, output=True)

In [ ]:
# TODO: change neuron type es. RLeaky https://snntorch.readthedocs.io/en/latest/snn.neurons_rleaky.html <Optional>

# first_layer_neuron = ...
# second_layer_neuron =  ...

In [39]:
# standard network with population coding

net_pop = nn.Sequential(nn.Flatten(),
                        nn.Linear(num_inputs, num_hidden),
                        first_layer_neuron,
                        nn.Linear(num_hidden, pop_outputs),
                        second_layer_neuron
                        ).to(device)

## Next Step: Define your own network with population coding

In [33]:
# TODO: define your own network (If you need another layer you need to define it. You can change the type of neurons between different layers)
# A full list of neurons type is available at https://snntorch.readthedocs.io/en/latest/snntorch.html#neuron-list
# Execute this step after reporting the results of the previous standard network with population coding

net = nn.Sequential(
    nn.Flatten(),
    nn.Linear(num_inputs, 128),
    snn.Leaky(beta=beta, spike_grad=grad, init_hidden=True),
    nn.Linear(128, 256),
    snn.Leaky(beta=beta, spike_grad=grad, init_hidden=True),
    nn.Linear(256, pop_outputs),
    snn.Leaky(beta=beta, spike_grad=grad, init_hidden=True, output=True)
).to(device)

# Training
## Without population coding
Define the optimizer and loss function. Here, we use the MSE Count Loss, which counts up the total number of output spikes at the end of the simulation run.

The correct class has a target firing probability of 100%, and incorrect classes are set to 0%.

In [29]:
#TODO: Select a random configuration of parameters
import snntorch.functional as SF

learning_rate = 2e-3 #GROUPS : A: [1e-3], B: [1.5e-3], C: [2e-3], D: [2.5e-3]

optimizer = torch.optim.Adam(net.parameters(), lr=learning_rate, betas=(0.9, 0.999))
loss_fn = SF.mse_count_loss(correct_rate=1.0, incorrect_rate=0.0)

We will also define a simple test accuracy function that predicts the correct class based on the neuron with the highest spike count.

In [25]:
from snntorch import utils

def test_accuracy(data_loader, net, num_steps, population_code=False, num_classes=False):
  with torch.no_grad():
    total = 0
    acc = 0
    net.eval()

    data_loader = iter(data_loader)
    for data, targets in data_loader:
      data = data.to(device)
      targets = targets.to(device)
      utils.reset(net)
      spk_rec, _ = net(data)

      if population_code:
        acc += SF.accuracy_rate(spk_rec.unsqueeze(0), targets, population_code=True, num_classes=n_classes) * spk_rec.size(1)
      else:
        acc += SF.accuracy_rate(spk_rec.unsqueeze(0), targets) * spk_rec.size(1)

      total += spk_rec.size(1)

  net.train()
  return acc/total

Let's run the training loop.

In [24]:
from snntorch import backprop

num_epochs = 20

# training loop
for epoch in range(num_epochs):

    avg_loss = backprop.BPTT(net, train_loader, num_steps=num_steps,
                          optimizer=optimizer, criterion=loss_fn, time_var=False, device=device)

    print(f"Epoch: {epoch}")
    print(f"Test set accuracy: {test_accuracy(test_loader, net, num_steps)*100:.3f}%\n")

/tmp/ipykernel_2047/2440134412.py:1: DeprecationWarning: The module snntorch.backprop will be deprecated in  a future release. Writing out your own training loop will lead to substantially faster performance.
  from snntorch import backprop


Epoch: 0
Test set accuracy: 6.141%

Epoch: 1
Test set accuracy: 9.006%

Epoch: 2
Test set accuracy: 9.070%

Epoch: 3
Test set accuracy: 12.022%

Epoch: 4
Test set accuracy: 13.844%

Epoch: 5
Test set accuracy: 16.199%

Epoch: 6
Test set accuracy: 15.027%

Epoch: 7
Test set accuracy: 16.030%

Epoch: 8
Test set accuracy: 13.112%

Epoch: 9
Test set accuracy: 17.772%

Epoch: 10
Test set accuracy: 17.713%

Epoch: 11
Test set accuracy: 18.195%

Epoch: 12
Test set accuracy: 20.544%

Epoch: 13
Test set accuracy: 18.016%

Epoch: 14
Test set accuracy: 19.947%

Epoch: 15
Test set accuracy: 19.194%

Epoch: 16
Test set accuracy: 20.946%

Epoch: 17
Test set accuracy: 16.280%

Epoch: 18
Test set accuracy: 24.527%

Epoch: 19
Test set accuracy: 17.680%



## With population coding

In [40]:
# TODO: Select a random configuration of parameters
import snntorch.functional as SF


learning_rate = 2e-3  #GROUPS : A: [1e-3], B: [1.5e-3], C: [2e-3], D: [2.5e-3]

loss_fn = SF.mse_count_loss(correct_rate=1.0, incorrect_rate=0.0, population_code=True, num_classes=n_classes)
optimizer = torch.optim.Adam(net_pop.parameters(), lr=learning_rate, betas=(0.9, 0.999))

In [41]:
from snntorch import backprop

num_epochs = 20

# training loop
for epoch in range(num_epochs):

    avg_loss = backprop.BPTT(net_pop, train_loader, num_steps=num_steps,
                            optimizer=optimizer, criterion=loss_fn, time_var=False, device=device)

    print(f"Epoch: {epoch}")
    print(f"Test set accuracy: {test_accuracy(test_loader, net_pop, num_steps, population_code=True, num_classes=n_classes)*100:.3f}%\n")

Epoch: 0
Test set accuracy: 47.633%

Epoch: 1
Test set accuracy: 51.617%

Epoch: 2
Test set accuracy: 54.730%

Epoch: 3
Test set accuracy: 55.292%

Epoch: 4
Test set accuracy: 57.604%

Epoch: 5
Test set accuracy: 58.590%

Epoch: 6
Test set accuracy: 59.247%

Epoch: 7
Test set accuracy: 60.263%

Epoch: 8
Test set accuracy: 58.839%

Epoch: 9
Test set accuracy: 60.668%

Epoch: 10
Test set accuracy: 61.669%

Epoch: 11
Test set accuracy: 62.013%

Epoch: 12
Test set accuracy: 61.012%

Epoch: 13
Test set accuracy: 61.935%

Epoch: 14
Test set accuracy: 62.465%

Epoch: 15
Test set accuracy: 63.231%

Epoch: 16
Test set accuracy: 63.341%

Epoch: 17
Test set accuracy: 64.467%

Epoch: 18
Test set accuracy: 63.168%

Epoch: 19
Test set accuracy: 65.234%



## Report of the parameters of configurations and results



| Classes | features | n_hidden | steps | beta | 1st neuron | 2nd neuron | pop_coding | neurons_per_class | lr | accuracy |
| -------- | ------ | -------- | -------- | -------- | -------- | -------- | -------- | -------- | -------- | -------- |
| 33 | 1881 | 256 | 20 | 0.85 | Leaky | Leaky | False | - | 2e-3 | 18% |
| 10 | 26 (PCA) | 256 | 20 | 0.85 | Leaky | Leaky | False | - | 2e-3 | 54% |
| 10 | 26 (PCA) | 256 | 20 | 0.85 | Leaky | Leaky | True | 75 | 2e-3 | 70% |
| 10 | 26 (PCA) | 256 | 20 | 0.85 | RLeaky | RLeaky | True | 75 | 2e-3 | 65% |

### Custom Network

| Classes | features | steps | beta | pop_coding | neurons_per_class | lr | accuracy |
| -------- | ------ |  -------- | -------- | -------- | -------- | -------- | -------- |
| 10 | 26 (PCA) |  20 | 0.85 | True | 50 | 2e-3 | 63% |
| 10 | 26 (PCA) |  20 | 0.85 | True | 75 | 2e-3 | 67% |

```
net = nn.Sequential(
    nn.Flatten(),
    nn.Linear(num_inputs, 128),
    snn.Leaky(beta=beta, spike_grad=grad, init_hidden=True),
    nn.Linear(128, 256),
    snn.Leaky(beta=beta, spike_grad=grad, init_hidden=True),
    nn.Linear(256, pop_outputs),
    snn.Leaky(beta=beta, spike_grad=grad, init_hidden=True, output=True)
).to(device)
```

# Conclusion
The performance boost from population coding may start to fade as the number of time steps increases. But it may also be preferable to increasing time steps as PyTorch is optimized for handling matrix-vector products, rather than sequential, step-by-step operations over time.

* For a detailed tutorial of spiking neurons, neural nets, encoding, and training using neuromorphic datasets, check out the
[snnTorch tutorial series](https://snntorch.readthedocs.io/en/latest/tutorials/index.html).
* For more information on the features of snnTorch, check out the [documentation at this link](https://snntorch.readthedocs.io/en/latest/).
* If you have ideas, suggestions or would like to find ways to get involved, then [check out the snnTorch GitHub project here.](https://github.com/jeshraghian/snntorch)